# RL basics: verifiable rewards, haiku edition

This tutorial uses Qwen3-4B and haiku poems to introduce the
**verifiable reward** pattern that underpins RL post-training:

1. Serve the base model.
2. Define a scoring function with a verifiable reward (syllable structure).
3. Evaluate the base model against that scorer.
4. GRPO-train the model with [slime](https://github.com/THUDM/slime) using the reward function.
5. Serve the trained checkpoint.
6. Evaluate it with the same scorer and compare.

**Why haikus?** A haiku has two attributes you can score
automatically — whether it follows the 5-7-5 syllable format
(deterministic, cheap) and whether the poem is actually good. That split between
*verifiable* and *subjective* rewards is exactly the landscape
RL post-training operates in. This tutorial covers the
verifiable half. In a later tutorial, we will cover the subjective half.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
import importlib.util

# Skip if modal_training_gym is already importable (e.g. a local editable
# checkout) so your edits keep taking effect and the env stays synced.
if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main
if importlib.util.find_spec('nltk') is None:
    %uv pip install -q nltk

In [ ]:
import re

from modal_training_gym import (
    DeploymentConfig,
    EvalConfig,
    EvalRowResult,
    HuggingFaceDataset,
    Qwen3_4B,
    SlimeRecipe,
    TrainConfig,
    list_checkpoints,
)

## Set up, serve, and evaluate the base model

So, how does Qwen3-4B currently fare at writing haikus? We can serve the base model and find out.

The training gym has several config classes so you can define deployment, training, and evaluation configurations, and
reuse them across different runs for parameter sweeps.

Let's start by creating a [`ModelConfig`](https://gym.modal.dev/reference/core/modelconfig) to define the base model. 
You can set one up yourself by subclassing `ModelConfig` or `HFModelConfiguration`, but we also provide pre-configured
ones for common models like `Qwen3_4B`.

Then, we can initialize a `DeploymentConfig` to serve our model. Calling `DeploymentConfig.serve()` builds and deploys an
SGLang app, then returns a `ModelDeployment` with the endpoint URL. For this tutorial, we pass `unauthenticated=True` so
the endpoint is reachable without proxy-auth tokens.

In [ ]:
base_model = Qwen3_4B()
base_model_deployment = DeploymentConfig(
    model=base_model,
    unauthenticated=True,
).serve()
print(f"Base model deployed to {base_model_deployment.url}")

The model will take a moment to download and spin up, but once it's ready, we can request it to write a haiku about a topic.

In [ ]:
response = base_model_deployment.generate(
    "Write a haiku about cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)

Chances are what it wrote was not, in fact, a haiku. But how can we determine this at scale? For that, we need a scoring
function.

A good eval takes a particular outcome and assigns a score to it. It can be binary (pass/fail) or continuous (0-100),
deterministic or subjective, and cheap or expensive to compute.

In our case, we want our model to be good at writing haiku poems. A haiku must follow the 5-7-5 syllable format, so we can
count syllables using NLTK's CMU Pronouncing Dictionary (with a regex fallback for words not in the dictionary) and score how
close each line is to its target syllable count.

We could give it score 0 if it doesn't follow the 5-7-5 syllable format, and 1 if it does. But that's not very informative.
Instead, we can score it based on how close it is to the target syllable count for each line.

In [ ]:
_cmudict_cache = {}

def _get_cmudict() -> dict:
    if not _cmudict_cache:
        import nltk
        from nltk.corpus import cmudict
        nltk.download("cmudict", quiet=True)
        _cmudict_cache.update(cmudict.dict())
    return _cmudict_cache

def _count_syllables(text: str) -> int:
    cmu = _get_cmudict()
    total = 0
    for word in re.findall(r"[a-zA-Z]+", text):
        phones = cmu.get(word.lower())
        if phones:
            total += sum(p[-1].isdigit() for p in phones[0])
        else:
            count = len(re.findall(r"[aeiouy]+", word.lower()))
            if word.lower().endswith("e") and count > 1:
                count -= 1
            total += max(count, 1)
    return total

def score_haiku(response: str) -> float:
    lines = [line.strip() for line in response.strip().split("\n") if line.strip()]
    if len(lines) != 3:
        return -10
    total_diff = sum(
        abs(_count_syllables(line) - target)
        for line, target in zip(lines, [5, 7, 5])
    )
    return -float(total_diff)

In [ ]:
response = base_model_deployment.generate(
    "Write a haiku about cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)
print(f"Score: {score_haiku(response)}")

Let's also define a dataset. Like models, you can subclass `DatasetConfig` to create a custom dataset, or `HuggingFaceDataset` to
automatically download one from Hugging Face.

Here, we use the statworx/haiku dataset from HuggingFace. Each row has a `keywords` topic and a reference `text` haiku. We can use
this dataset to train our model.

Datasets for training models can take many form factors; this is just one of them. If you're curious about other options, check
out the [DatasetConfig](https://gym.modal.dev/reference/core/datasetconfig/) documentation.

In [ ]:
class HaikuDataset(HuggingFaceDataset):
    hf_repo = "statworx/haiku"
    input_column = "keywords"
    output_column = "text"
    output_format = "jsonl"
    apply_chat_template = True
    system_prompt = (
        "You are a haiku poet. Write a haiku about the given topic. "
        "Use the 5-7-5 syllable format across three lines."
    )
    prompt_template = "Write a haiku about {input}."
    always_prepare = True # For the purpose of this tutorial, we want to prepare the dataset every time we run it, in case there is stale data from a previous run.

train_dataset = HaikuDataset(n_rows=10)
eval_dataset = HaikuDataset(n_rows=5)

Let's take a quick peek into the eval set:

In [ ]:
df = eval_dataset.to_pandas()
print(len(df))
df.head(5)

Seems straightforward enough, right? How do we run an eval on our base model with this dataset? We can transform our scoring
function above into an [`EvalConfig`](https://gym.modal.dev/reference/evaluation/evalconfig).

An EvalConfig is a class that owns the model-calling loop. To specify the task, you pass in a dataset and one of two things:
1. An `eval_response_fn`. EvalConfig will generate a response for each row in the dataset and pass it to your function. You
   then calculate a score and return it in an `EvalRowResult`.
2. For more complex evals (e.g. multi-turn), you can define a custom `EvalConfig.eval_fn` that takes a `ModelDeployment` and
   a dataset row and returns a score.

Once set up, you can call `.evaluate(...)` with your deployed model to run the eval. Let's see how our base model fares:

In [ ]:
def eval_response_fn(_example: dict, response: str) -> EvalRowResult:
    return EvalRowResult(score=score_haiku(response), response=response)

eval_config = EvalConfig(
    dataset=eval_dataset,
    eval_response_fn=eval_response_fn,
    generate_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
)
print("——— Running base model evaluation... ———")
base_eval = eval_config.evaluate(base_model_deployment, debug=True)
print(f"Average haiku score: {base_eval.mean:.1f}")
print("——— Base model evaluation complete ———")

## Train with slime

Now, let's actually train the model to write good haikus.
Here, we use the [slime](https://github.com/THUDM/slime) framework on Modal.

To train a model on the gym, you create a [`TrainConfig`](https://gym.modal.dev/reference/training/trainconfig/) object,
which takes a model, dataset, and *recipe*. A recipe lets you configure the training loop, including the reward function,
how many rollouts to run, GPU configuration, and more.

All flags that are native to slime can be passed to the [`SlimeRecipe`](https://gym.modal.dev/reference/training/slimerecipe/)
object. You can also customize the training environment or add patches to slime using the `image_overlay` argument.

In [ ]:
async def haiku_rm(args, sample, **kwargs) -> float:
    response = base_model.parse_response(sample.response)
    return score_haiku(response.content)

training_run = TrainConfig(
    model=base_model,
    dataset=train_dataset,
    recipe=SlimeRecipe(
        custom_rm_function=haiku_rm,

        gpu_type="H100",
        colocate=True,
        tensor_model_parallel_size=1,
        sequence_parallel=False,
        rollout_num_gpus_per_engine=1,

        num_rollout=10,
        rollout_batch_size=16,
        rollout_max_response_len=4096,
        rollout_temperature=1.0,

        save_interval=5,
        apply_chat_template_kwargs='{"enable_thinking": false}',

        image_overlay=lambda image: image.run_commands(
            "uv pip install --system aiohttp nltk>=3.8.0",
            "python -c \"import nltk; nltk.download('cmudict', quiet=True)\"",
        ),
    ),
)

Once we have a `TrainConfig`, all that's left to do is call `.train()`! This will build a Modal app, print out a dashboard
URL, and run training. All results will be stored in a Modal volume for later use.

This will take some time, but while it's running, you can open the dashboard URL in your browser to see progress and even
samples from each of the rollouts. For more details, check out the [observability dashboard docs](https://gym.modal.dev/tutorials/tools/000_observability_dashboard).

In [ ]:
print("——— Running training... ———")
train_result = training_run.train()
print("——— Training complete ———")

## Serve and evaluate the trained checkpoint

The returned `TrainResult` has the checkpoint path and volume metadata attached. To serve our trained model, we can pass
this checkpoint to a new `DeploymentConfig`:

In [ ]:
checkpoint = list_checkpoints(train_result.training_run_id)[-1]
print(checkpoint.path)

trained_model_deployment = DeploymentConfig(
    model=Qwen3_4B(),
    checkpoint=checkpoint,
    app_name="qwen3-4b-haiku-serve",
    served_model_name="qwen3-4b-haiku",
    unauthenticated=True,
).serve()
print(f"Trained model deployed to {trained_model_deployment.url}")

Now let's run the same eval on the trained model and compare.

In [ ]:
print("——— Running trained model evaluation... ———")
trained_eval = eval_config.evaluate(trained_model_deployment, debug=True)
print(f"Trained haiku score: {trained_eval.mean:.1f}")
print("——— Trained model evaluation complete ———")

## Train off of a checkpoint
Hmm, looks like the trained model could still do better.
Maybe it's because it only trained for 10 iterations.

What happens if we train it for more?
We want to train it off of the latest checkpoint, not from scratch.
To do this, we make a new `TrainConfig`, this time with our checkpoint:

In [ ]:
new_training_run = TrainConfig(
    model=Qwen3_4B(),
    dataset=train_dataset,
    checkpoint=checkpoint,
    recipe=SlimeRecipe(
        custom_rm_function=haiku_rm,

        gpu_type="H100",
        colocate=True,
        tensor_model_parallel_size=1,
        sequence_parallel=False,
        rollout_num_gpus_per_engine=1,

        num_rollout=20,
        rollout_batch_size=16,
        rollout_max_response_len=4096,
        rollout_temperature=1.0,

        save_interval=10,
        apply_chat_template_kwargs='{"enable_thinking": false}',

        image_overlay=lambda image: image.run_commands(
            "uv pip install --system aiohttp nltk>=3.8.0",
            "python -c \"import nltk; nltk.download('cmudict', quiet=True)\"",
        ),
    ),
)
print("——— Running new training... ———")
new_train_result = new_training_run.train()
print("——— New training complete ———")

## Evaluate the continued checkpoint

Now let's run the same eval on the newly trained model and compare.

In [ ]:
new_checkpoint = list_checkpoints(new_train_result.training_run_id)[-1]
print(new_checkpoint.path)

new_model_deployment = DeploymentConfig(
    model=Qwen3_4B(),
    checkpoint=new_checkpoint,
    app_name="qwen3-4b-haiku-serve-new",
    served_model_name="qwen3-4b-haiku",
    unauthenticated=True,
).serve()
print(f"Newly trained model deployed to {new_model_deployment.url}")

Now let's compare the results of the newly trained model and the base model.

In [ ]:
print("——— Running trained model evaluation... ———")
new_eval = eval_config.evaluate(new_model_deployment, debug=True)
print(f"Trained model (new) haiku score: {new_eval.mean:.1f}")
print("——— Trained model (new) evaluation complete ———")

Now let's compare the results across all three checkpoints.

In [ ]:
print(f"Base model haiku score: {base_eval.mean:.1f}")
print(f"Trained model haiku score: {trained_eval.mean:.1f}")
print(f"Trained model (new) haiku score: {new_eval.mean:.1f}")

## Conclusion

We've seen how to use the gym to serve a base model, define a scoring function, evaluate it, train a model, and serve the
trained model. Hopefully, you got some good haikus out of it too!

This is a simple example, but it demonstrates the core concepts of RL post-training. There are still many more things you
can do with the gym, like:

* [Code execution in your reward function using Modal Sandboxes](https://gym.modal.dev/tutorials/rl/001_sandboxes/)
* [Multi-turn RL](https://gym.modal.dev/tutorials/rl/002_multiturn/)
* [On-policy distillation](https://gym.modal.dev/tutorials/rl/003_on_policy_distillation/)
* [DAPO for complex reasoning tasks](https://gym.modal.dev/tutorials/rl/005_dapo/)
* [Audio](https://gym.modal.dev/tutorials/rl/006_audio_asr/) and [computer use](https://gym.modal.dev/tutorials/rl/008_computer_use/) models
* [Parameter sweeps to easily tune hyperparameters](https://gym.modal.dev/tutorials/rl/007_param_sweep/)
* [Cross-tokenizer distillation for training across model families](https://gym.modal.dev/tutorials/rl/009_cross_tokenizer_distillation/)